In [1]:
!git clone https://github.com/Ulyssesllc/ulysses-gpt.git
%cd /kaggle/working/ulysses-gpt
!pip install -e ".[dev]"


Cloning into 'ulysses-gpt'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 211 (delta 114), reused 164 (delta 67), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 78.36 KiB | 2.70 MiB/s, done.
Resolving deltas: 100% (114/114), done.
/kaggle/working/ulysses-gpt
Obtaining file:///kaggle/working/ulysses-gpt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.7/531.7 kB 25.3 MB/s eta 0:00:00
  Building editable for ulysses-gpt (pyproject.toml)

In [2]:
!ls -R

.:
check_code.sh  Dockerfile  pyproject.toml  requirements.txt  src    train.py
configs        LICENSE	   README.md	   run_project.sh    tests

./configs:
training.env.example

./src:
mini_gpt  ulysses_gpt.egg-info

./src/mini_gpt:
dataset.py   hf_export.py  metrics.py  post_train.py
generate.py  __init__.py   model.py    trainer.py

./src/ulysses_gpt.egg-info:
dependency_links.txt  PKG-INFO	    SOURCES.txt
entry_points.txt      requires.txt  top_level.txt

./tests:
test_model.py  test_post_train.py  test_trainer.py


In [3]:
!bash check_code.sh

🔍 [1/3] Running Ruff Linter & Formatter...
All checks passed!
13 files already formatted
✓ Ruff checks passed!

🔬 [2/3] Running Mypy Static Type Checker...
Success: no issues found in 8 source files
✓ Mypy type check passed!

🧪 [3/3] Running Pytest Unit Tests...
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/ulysses-gpt
configfile: pyproject.toml
plugins: cov-7.1.0, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 15 items                                                             

tests/test_model.py::test_gpt_config_defaults PASSED                     [  6%]
tests/test_model.py::test_causal_self_attention_shape PASSED             [ 13%]
tests/test_model.py::test_mlp_shape PASSED                               [ 20%]
tests/test_model.py::test_transformer_block_shape PASSED                 [ 26%]
tests/test_model.p

In [4]:
import os

os.environ.update({
    # Pretraining: WikiText-2
    "PRETRAIN_HF_DATASET": "Salesforce/wikitext",
    "PRETRAIN_HF_CONFIG": "wikitext-2-raw-v1",
    "PRETRAIN_HF_SPLIT": "train",
    "PRETRAIN_VALIDATION_SPLIT": "validation",
    "TEXT_COLUMN": "text",

    # Đường dẫn checkpoint
    "CHECKPOINT_DIR": "/kaggle/working/ulysses-gpt/checkpoints",

    # Cấu hình pretraining
    "BLOCK_SIZE": "512",
    "BATCH_SIZE": "1",
    "GRAD_ACCUM_STEPS": "16",
    "N_LAYER": "8",
    "N_HEAD": "8",
    "N_EMBD": "512",
    "EPOCHS": "3",
    "MAX_STEPS": "10000",
    "MAX_TOKENS":"20000000",
    "MAX_VAL_TOKENS":"2000000",
    "LEARNING_RATE": "0.0002",
    "SEED": "42",
    "PROMPT": "Explain the main idea of this passage:",

    # SFT: Dolly-15k
    "SFT_HF_DATASET": "databricks/databricks-dolly-15k",
    "SFT_HF_SPLIT": "train",
    "SFT_PROMPT_COLUMN": "instruction",
    "SFT_CONTEXT_COLUMN": "context",
    "RESPONSE_COLUMN": "response",
    "SFT_EPOCHS": "1",
    "SFT_MAX_SAMPLES": "3000",
    "SFT_LR": "0.00005",

    # DPO
    "DPO_HF_DATASET": "CultriX/preference-dataset",
    "DPO_HF_SPLIT": "train",
    "PROMPT_COLUMN": "prompt",
    "CHOSEN_COLUMN": "chosen",
    "REJECTED_COLUMN": "rejected",
    "DPO_EPOCHS": "1",
    "DPO_MAX_SAMPLES": "1000",
    "DPO_LR": "0.000005",
})

!bash run_project.sh

Stage 1/3: pretraining
Running on device: CUDA
Total parameters: 51,213,824
Trainable parameters: 51,213,824
Loading Hugging Face dataset 'Salesforce/wikitext' (split='train')...
README.md: 100%|███████████████████████████| 10.5k/10.5k [00:00<00:00, 21.4MB/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:  |  0.00B       
wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes: █|  732kB, 72.6k
wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file: 100%|█|  733kB

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:  |  415kB       
wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes: █| 6.35MB,  630k
wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file: 100%|█| 6.36MB

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:  |  0.00B       
wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes: █|  657kB, 65.4k
wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file: 100%|█|  657kB
Generati